# IVECO Statistics - Modular VODR Pipeline

Notebook VODR diviso in step, cosi' ogni fase lascia variabili intermedie ispezionabili.

In [ ]:
import os

import pyspark.sql.functions as F

from run_local_sample import build_spark, ensure_excel_writer_available, export_excel_outputs
from engine_cleaning import add_legacy_preparation_features, clean_spark_column_names, keep_latest_record_per_vin
from engine_loader import extract_metadata
from engine_utils import log_step, report_dim, report_vin
from vodr_config import VODR_EXCLUDED_VINS, config_mt_from_vodr, get_vodr_export_file_name, get_vodr_report_sheets
from vodr_pipeline import (
    DEFAULT_VODR_CONFIG,
    DEFAULT_VODR_INPUT_MODE,
    DEFAULT_VODR_OUTPUT_DIR,
    DEFAULT_VODR_SAMPLE_FORMAT,
    DEFAULT_VODR_SAMPLE_PATH,
    add_vodr_time_percentages,
    build_vodr_sheet_output,
    import_mission_test_statistics,
    parse_config_text,
    prepare_mission_test_join_frame,
    read_vodr_input_data,
)

## 0. Widget e configurazione

In [ ]:
default_config_text = ','.join(map(str, sorted(DEFAULT_VODR_CONFIG)))

try:
    dbutils
    try:
        dbutils.widgets.get("config")
    except Exception:
        dbutils.widgets.text("config", default_config_text, "Config VODR")
    try:
        dbutils.widgets.get("input_mode")
    except Exception:
        dbutils.widgets.dropdown("input_mode", DEFAULT_VODR_INPUT_MODE, ["fat_table", "sample"], "Input mode")
    try:
        dbutils.widgets.get("join_mission_test")
    except Exception:
        dbutils.widgets.dropdown("join_mission_test", "Yes", ["Yes", "No"], "Join Mission Test")
    try:
        dbutils.widgets.get("join_type")
    except Exception:
        dbutils.widgets.dropdown("join_type", "inner", ["inner", "left"], "Join type")
    try:
        dbutils.widgets.get("include_complete_dataset")
    except Exception:
        dbutils.widgets.dropdown("include_complete_dataset", "Yes", ["Yes", "No"], "Complete Dataset")
    try:
        dbutils.widgets.get("export_excel")
    except Exception:
        dbutils.widgets.dropdown("export_excel", "Yes", ["Yes", "No"], "Export Excel")
    try:
        dbutils.widgets.get("output_dir")
    except Exception:
        dbutils.widgets.text("output_dir", DEFAULT_VODR_OUTPUT_DIR, "Output dir")
    try:
        dbutils.widgets.get("sample_path")
    except Exception:
        dbutils.widgets.text("sample_path", DEFAULT_VODR_SAMPLE_PATH, "Sample path")
    try:
        dbutils.widgets.get("input_format")
    except Exception:
        dbutils.widgets.dropdown("input_format", DEFAULT_VODR_SAMPLE_FORMAT, ["parquet", "csv"], "Sample format")

    config_text = dbutils.widgets.get("config")
    input_mode = dbutils.widgets.get("input_mode")
    join_mission_test = dbutils.widgets.get("join_mission_test") == "Yes"
    join_type = dbutils.widgets.get("join_type")
    include_complete_dataset = dbutils.widgets.get("include_complete_dataset") == "Yes"
    export_excel = dbutils.widgets.get("export_excel") == "Yes"
    output_dir = dbutils.widgets.get("output_dir")
    sample_path = dbutils.widgets.get("sample_path")
    input_format = dbutils.widgets.get("input_format")
except NameError:
    config_text = default_config_text
    input_mode = DEFAULT_VODR_INPUT_MODE
    join_mission_test = True
    join_type = "inner"
    include_complete_dataset = True
    export_excel = True
    output_dir = DEFAULT_VODR_OUTPUT_DIR
    sample_path = DEFAULT_VODR_SAMPLE_PATH
    input_format = DEFAULT_VODR_SAMPLE_FORMAT

config = parse_config_text(config_text)
config_mt = config_mt_from_vodr(config)

log_step(f"Config VODR selezionate: {sorted(config)}")
log_step(f"Config Mission Test collegate: {sorted(config_mt)}")
log_step(f"Input mode: {input_mode}")
log_step(f"Join Mission Test: {join_mission_test} ({join_type})")
log_step(f"Complete Dataset: {include_complete_dataset}")
log_step(f"Export Excel: {export_excel}")
log_step(f"Output dir: {output_dir}")

## 1. Setup Spark e dipendenze Excel

In [ ]:
try:
    spark
except NameError:
    spark = build_spark("iveco-vodr-pipeline")

if export_excel:
    excel_engine = ensure_excel_writer_available(auto_install=True)
    log_step(f"Engine Excel disponibile: {excel_engine}")
else:
    excel_engine = None
    log_step("Export Excel disabilitato")

## 2. Lettura input VODR

In [ ]:
df_vodr_raw = read_vodr_input_data(
    spark=spark,
    input_mode=input_mode,
    sample_path=sample_path,
    input_format=input_format,
    config=config,
)

report_dim(df_vodr_raw, "VODR raw")
report_vin(df_vodr_raw)

try:
    display(df_vodr_raw.limit(10))
except NameError:
    df_vodr_raw.show(10, truncate=False)

## 3. Cleaning VODR e ultimo record per VIN

In [ ]:
df_vodr_clean = clean_spark_column_names(df_vodr_raw)

if "vin" in df_vodr_clean.columns:
    df_vodr_clean = df_vodr_clean.filter(~F.col("vin").isin(VODR_EXCLUDED_VINS))

df_vodr_latest = keep_latest_record_per_vin(df_vodr_clean)

report_dim(df_vodr_latest, "VODR latest VIN")
report_vin(df_vodr_latest)

try:
    display(df_vodr_latest.limit(10))
except NameError:
    df_vodr_latest.show(10, truncate=False)

## 4. Lettura Mission Test per join

In [ ]:
if join_mission_test and input_mode == "fat_table":
    df_mt_raw = import_mission_test_statistics(spark, config_mt)
    report_dim(df_mt_raw, "Mission Test raw")
    report_vin(df_mt_raw)

    df_mt_join = prepare_mission_test_join_frame(df_mt_raw)
    report_dim(df_mt_join, "Mission Test latest VIN join frame")
    report_vin(df_mt_join)

    try:
        display(df_mt_join.limit(10))
    except NameError:
        df_mt_join.show(10, truncate=False)
else:
    df_mt_raw = None
    df_mt_join = None
    log_step("Join Mission Test saltato")

## 5. Join VODR + Mission Test

In [ ]:
if df_mt_join is not None:
    df_vodr_joined = df_vodr_latest.join(df_mt_join, on="vin", how=join_type)

    if "udt_timestamp" in df_vodr_joined.columns and "udt_timestamp_mt" in df_vodr_joined.columns:
        date_filter = F.col("_vodr_mt_day_diff").between(-1, 1)
        if join_type == "left":
            date_filter = date_filter | F.col("udt_timestamp_mt").isNull()

        df_vodr_joined = (
            df_vodr_joined.withColumn(
                "_vodr_mt_day_diff",
                F.datediff(F.to_date(F.col("udt_timestamp_mt")), F.to_date(F.col("udt_timestamp"))),
            )
            .filter(date_filter)
            .drop("_vodr_mt_day_diff")
        )

    if "Average_vehicle_speed_mt" in df_vodr_joined.columns:
        if "Average_vehicle_speed" in df_vodr_joined.columns:
            df_vodr_joined = df_vodr_joined.withColumn(
                "Average_vehicle_speed",
                F.coalesce(F.col("Average_vehicle_speed"), F.col("Average_vehicle_speed_mt")),
            )
        else:
            df_vodr_joined = df_vodr_joined.withColumn("Average_vehicle_speed", F.col("Average_vehicle_speed_mt"))

    if "mileage_mt" in df_vodr_joined.columns:
        if "mileage" in df_vodr_joined.columns:
            df_vodr_joined = df_vodr_joined.withColumn("mileage", F.coalesce(F.col("mileage"), F.col("mileage_mt")))
        else:
            df_vodr_joined = df_vodr_joined.withColumn("mileage", F.col("mileage_mt"))
else:
    df_vodr_joined = df_vodr_latest

report_dim(df_vodr_joined, "VODR joined")
report_vin(df_vodr_joined)

if "id_config" in df_vodr_joined.columns:
    df_vodr_joined.groupBy("id_config").count().orderBy("id_config").show()
if "id_config_mt" in df_vodr_joined.columns:
    df_vodr_joined.groupBy("id_config_mt").count().orderBy("id_config_mt").show()

## 6. Arricchimento colonne legacy Prep

In [ ]:
df_vodr_prepared = add_legacy_preparation_features(df_vodr_joined)

if "TotEngineHours" in df_vodr_prepared.columns and "engineminutes" not in df_vodr_prepared.columns:
    df_vodr_prepared = df_vodr_prepared.withColumn("engineminutes", F.col("TotEngineHours").cast("double") * F.lit(60))

p_type, p_group, p_series = extract_metadata(df_vodr_prepared)
log_step(f"Metadata rilevati: type={p_type}, group={p_group}, series={p_series}")
report_dim(df_vodr_prepared, "VODR prepared")

try:
    display(df_vodr_prepared.limit(10))
except NameError:
    df_vodr_prepared.show(10, truncate=False)

## 7. Percentuali VODR

In [ ]:
df_time_percentage = add_vodr_time_percentages(df_vodr_prepared, config)
report_dim(df_time_percentage, "VODR time_percentage")

try:
    display(df_time_percentage.limit(10))
except NameError:
    df_time_percentage.show(10, truncate=False)

## 8. Helper preview sheet

In [ ]:
vodr_sheet_settings = {sheet["sheet_id"]: sheet for sheet in get_vodr_report_sheets(config)}
vodr_sheet_ids = list(vodr_sheet_settings.keys())
sheet_outputs_by_id = {}

if not include_complete_dataset and "complete_dataset" in vodr_sheet_ids:
    vodr_sheet_ids = [sheet_id for sheet_id in vodr_sheet_ids if sheet_id != "complete_dataset"]

log_step(f"Sheet VODR configurati: {vodr_sheet_ids}")

def preview_vodr_sheet(sheet_id):
    sheet = vodr_sheet_settings[sheet_id]
    log_step(f"Preview VODR {sheet_id} - {sheet['name']}")

    try:
        output = build_vodr_sheet_output(df_time_percentage, sheet)
    except Exception as exc:
        print(f"Sheet VODR {sheet_id} in errore: {exc}")
        return None

    if output is None:
        print(f"Sheet VODR saltato: {sheet_id} - {sheet['name']}")
        return None

    sheet_outputs_by_id[sheet_id] = output
    df_sheet = output["dataframe"]

    if getattr(df_sheet, "empty", False):
        print(f"Sheet VODR vuoto: {sheet_id} - {sheet['name']}")
        return df_sheet

    try:
        display(df_sheet)
    except NameError:
        print(df_sheet.head(20).to_string(index=False))

    return df_sheet

# Sheet Excel

## Complete Dataset

In [ ]:
if include_complete_dataset:
    preview_vodr_sheet("complete_dataset")
else:
    log_step("Complete Dataset saltato da widget")

## 1b Accelerator Pedal Position

In [ ]:
preview_vodr_sheet("1b")

In [ ]:
preview_vodr_sheet("1b_2")

In [ ]:
preview_vodr_sheet("1b_3")

## 2a Vehicle Weight Time

In [ ]:
preview_vodr_sheet("2a")

## 2b Vehicle Speed

In [ ]:
preview_vodr_sheet("2b")

In [ ]:
preview_vodr_sheet("2b_2")

## 2c External Temperature

In [ ]:
preview_vodr_sheet("2c")

## 3a Brake Pedal Position

In [ ]:
preview_vodr_sheet("3a")

In [ ]:
preview_vodr_sheet("3a_2")

## 3b Drive Retarder Torque

In [ ]:
preview_vodr_sheet("3b")

## 3c Retarder Oil Temperature

In [ ]:
preview_vodr_sheet("3c")

## 3d Retarder Water Temperature

In [ ]:
preview_vodr_sheet("3d")

## 4a Gear Shift

In [ ]:
preview_vodr_sheet("4a")

## 4c Transmission Mode

In [ ]:
preview_vodr_sheet("4c")

In [ ]:
preview_vodr_sheet("4c_2")

## 5a Battery Voltage

In [ ]:
preview_vodr_sheet("5a")

## 5b Battery SOC

In [ ]:
preview_vodr_sheet("5b")

## 5c Battery SOH

In [ ]:
preview_vodr_sheet("5c")

In [ ]:
preview_vodr_sheet("5c_2")

## 6a Cranking Time

In [ ]:
preview_vodr_sheet("6a")

## 6b Current during Cranking

In [ ]:
preview_vodr_sheet("6b")

In [ ]:
preview_vodr_sheet("6b_2")

## 6c Voltage during Cranking

In [ ]:
preview_vodr_sheet("6c")

In [ ]:
preview_vodr_sheet("6c_2")

## 6d Current measured

In [ ]:
preview_vodr_sheet("6d")

## Driving Style

In [ ]:
preview_vodr_sheet("average_kick_down")

In [ ]:
preview_vodr_sheet("average_kick_down_2")

In [ ]:
preview_vodr_sheet("selection_mode")

In [ ]:
preview_vodr_sheet("selection_mode_2")

In [ ]:
preview_vodr_sheet("time_in_semi")

In [ ]:
preview_vodr_sheet("aebs_intervention")

## Safety, Level, Lights

In [ ]:
preview_vodr_sheet("safety")

In [ ]:
preview_vodr_sheet("level")

In [ ]:
preview_vodr_sheet("lights")

## Export Excel

In [ ]:
if export_excel:
    ordered_sheet_outputs = [
        sheet_outputs_by_id[sheet_id]
        for sheet_id in vodr_sheet_ids
        if sheet_id in sheet_outputs_by_id
    ]

    excel_file_name = get_vodr_export_file_name(config)
    vodr_excel_path = export_excel_outputs(ordered_sheet_outputs, output_dir, excel_file_name)
    log_step(f"Excel generato: {vodr_excel_path}")
else:
    ordered_sheet_outputs = []
    vodr_excel_path = None
    log_step("Export Excel disabilitato")

## Copia Excel su DBFS

In [ ]:
if vodr_excel_path:
    try:
        dbutils
        local_excel_path = os.path.abspath(str(vodr_excel_path))
        dbfs_output_dir = "dbfs:/FileStore/iveco_vodr_output"
        dbfs_excel_path = f"{dbfs_output_dir}/{os.path.basename(local_excel_path)}"

        dbutils.fs.mkdirs(dbfs_output_dir)
        dbutils.fs.cp(f"file:{local_excel_path}", dbfs_excel_path, True)
        log_step(f"Excel copiato su DBFS: {dbfs_excel_path}")

        workspace_url = spark.conf.get("spark.databricks.workspaceUrl", None)
        if workspace_url:
            print(f"Download: https://{workspace_url}/files/iveco_vodr_output/{os.path.basename(local_excel_path)}")
    except NameError:
        log_step("Esecuzione non Databricks: copia DBFS saltata")
else:
    log_step("Nessun Excel da copiare su DBFS")

In [ ]:
log_step("Main pipeline VODR modulare completata")